# Lab 8 — MCP-Style Standardised Tool Layer
**Pattern: MODEL CONTEXT PROTOCOL** — from HTML Section 08

---

## The problem MCP solves

Before MCP, every AI application wrote its own connector to every tool.

```
3 AI apps × 3 tools = 9 custom integrations
```

Each one bespoke. Each one needing separate maintenance. Change Slack's API → fix 3 connectors.

MCP is the standard protocol that reduces this to:

```
3 AI apps + 3 tools = 6 implementations
```

Build the Slack connector once as an MCP server. Every agent connects to it the same way.  
Change the Slack connector → update one server. All agents inherit it automatically.

---

## Real-world analogy

**USB-C.**

Before USB-C, every device had a different cable. Building an adapter for each device-to-device combination was a full-time job.

USB-C is the standard socket. Your agent is the device. The MCP server is the adapter. One standard, universal connection.

---

## What this lab builds

A `ToolRegistry` class that IS the MCP server concept in Python:
- Tools register themselves once
- Agents ask the registry for schemas and execute via a standard interface
- Every tool call is logged automatically
- Swap an implementation → zero agent code changes

---

## What you will learn
1. The N×M integration problem and how a registry solves it
2. How to decouple agent logic from tool implementations completely
3. How to build automatic observability into every tool call
4. How error handling at the registry level protects agents from crashes

## Step 1 — Setup

In [ ]:
import anthropic
from typing import Callable
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()
print("✓ Client ready")

## Step 2 — The Tool Registry

This is the centrepiece of the lab.

The `ToolRegistry` is the **MCP Server** concept in Python code:
- `register()` → tools advertise their capabilities (one time, at startup)
- `get_tool_schemas()` → agents fetch all available tools in Anthropic API format
- `execute()` → standard interface for running any tool by name
- `show_call_log()` → automatic audit trail of every tool invocation

In [ ]:
class ToolRegistry:
    """
    Central registry for all tools.
    
    The agent only talks to the registry — never to tools directly.
    This is the MCP Server concept: a standardised layer between
    the agent (host) and the tools (data sources).
    """

    def __init__(self):
        self._tools: dict[str, dict] = {}
        self._call_log: list[dict]   = []

    def register(self, name: str, description: str, schema: dict, fn: Callable):
        """Register a tool. Called once at startup for each tool."""
        self._tools[name] = {"description": description, "schema": schema, "fn": fn}
        print(f"  [REGISTRY] Registered: '{name}'")

    def get_tool_schemas(self) -> list[dict]:
        """
        Returns all tool schemas in Anthropic API format.
        The agent passes this directly to client.messages.create(tools=...).
        The agent doesn't know what tools DO — just what they look like.
        """
        return [
            {"name": name, "description": t["description"], "input_schema": t["schema"]}
            for name, t in self._tools.items()
        ]

    def execute(self, name: str, inputs: dict) -> str:
        """
        Execute a tool by name via the standard interface.
        Logs every call. Returns errors as strings — not exceptions.
        Agents receive errors as tool results, not as crashes.
        """
        if name not in self._tools:
            error = f"Unknown tool: '{name}'. Available: {list(self._tools.keys())}"
            self._call_log.append({"tool": name, "inputs": inputs, "result": error, "status": "error"})
            return error
        try:
            result = str(self._tools[name]["fn"](**inputs))
            self._call_log.append({"tool": name, "inputs": inputs, "result": result, "status": "ok"})
            print(f"  [REGISTRY] ✓ {name}({inputs}) → {result[:70]}")
            return result
        except Exception as e:
            error = f"Tool '{name}' error: {str(e)}"
            self._call_log.append({"tool": name, "inputs": inputs, "result": error, "status": "error"})
            print(f"  [REGISTRY] ✗ {name} raised: {e}")
            return error

    def show_call_log(self):
        """Full audit trail. In production: CloudWatch, Datadog, Splunk."""
        print("\n── Tool Call Audit Log ──")
        for i, entry in enumerate(self._call_log, 1):
            icon = "✓" if entry["status"] == "ok" else "✗"
            print(f"  {i}. [{icon}] {entry['tool']}({entry['inputs']}) → {entry['result'][:60]}")

print("✓ ToolRegistry class defined")

## Step 3 — Tool Implementations

The underlying functions. The registry wraps them.

> Swap any implementation → update the registry → all agents using this registry  
> get the new version automatically. Zero agent code changes.

In [ ]:
def _get_weather(city: str, unit: str = "celsius") -> str:
    # Simulated — replace with OpenWeatherMap or WeatherAPI
    data = {
        "london":   {"temp": 14, "condition": "Cloudy",        "humidity": 78},
        "new york": {"temp": 22, "condition": "Sunny",         "humidity": 55},
        "tokyo":    {"temp": 28, "condition": "Humid",         "humidity": 85},
        "sydney":   {"temp": 18, "condition": "Partly cloudy", "humidity": 65},
        "dubai":    {"temp": 35, "condition": "Sunny",         "humidity": 40},
    }
    d    = data.get(city.lower(), {"temp": 20, "condition": "Unknown", "humidity": 60})
    temp = d["temp"] if unit == "celsius" else round(d["temp"] * 9/5 + 32, 1)
    unit_label = "°C" if unit == "celsius" else "°F"
    return f"{city}: {d['condition']}, {temp}{unit_label}, humidity {d['humidity']}%"


def _convert_units(value: float, from_unit: str, to_unit: str) -> str:
    conversions = {
        ("km",    "miles"):       lambda x: x * 0.621371,
        ("miles", "km"):          lambda x: x * 1.60934,
        ("kg",    "lbs"):         lambda x: x * 2.20462,
        ("lbs",   "kg"):          lambda x: x * 0.453592,
        ("celsius",    "fahrenheit"): lambda x: x * 9/5 + 32,
        ("fahrenheit", "celsius"):    lambda x: (x - 32) * 5/9,
    }
    key = (from_unit.lower(), to_unit.lower())
    fn  = conversions.get(key)
    if not fn:
        return f"No conversion for {from_unit} → {to_unit}. Supported: {list(conversions.keys())}"
    return f"{value} {from_unit} = {round(fn(value), 4)} {to_unit}"


def _lookup_timezone(city: str) -> str:
    zones = {
        "london":    "GMT+0 (BST GMT+1 in summer)",
        "new york":  "EST (UTC-5) / EDT (UTC-4) in summer",
        "tokyo":     "JST (UTC+9) — no daylight saving",
        "sydney":    "AEDT (UTC+11) / AEST (UTC+10) in winter",
        "dubai":     "GST (UTC+4) — no daylight saving",
        "singapore": "SGT (UTC+8) — no daylight saving",
    }
    return zones.get(city.lower(), f"Timezone for '{city}': not found")


def _get_exchange_rate(from_currency: str, to_currency: str, amount: float = 1.0) -> str:
    rates = {"USD": 1.0, "GBP": 0.79, "EUR": 0.92, "JPY": 149.5, "INR": 83.2, "AUD": 1.53}
    frm, to = from_currency.upper(), to_currency.upper()
    if frm not in rates or to not in rates:
        return f"Unknown currency. Supported: {list(rates.keys())}"
    converted = (amount / rates[frm]) * rates[to]
    return f"{amount} {frm} = {round(converted, 4)} {to}"

print("✓ Tool implementations defined")

## Step 4 — Build the Registry

Register all four tools. This happens **once at startup** — just like an MCP server advertising its capabilities.

In [ ]:
registry = ToolRegistry()

registry.register(
    name="get_weather",
    description="Get current weather for a city. Returns temperature, condition, humidity.",
    schema={
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name, e.g. London, Tokyo"},
            "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
        },
        "required": ["city"]
    },
    fn=_get_weather
)

registry.register(
    name="convert_units",
    description="Convert between units: km↔miles, kg↔lbs, celsius↔fahrenheit.",
    schema={
        "type": "object",
        "properties": {
            "value":     {"type": "number"},
            "from_unit": {"type": "string"},
            "to_unit":   {"type": "string"}
        },
        "required": ["value", "from_unit", "to_unit"]
    },
    fn=_convert_units
)

registry.register(
    name="lookup_timezone",
    description="Get timezone information for a major city.",
    schema={
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"]
    },
    fn=_lookup_timezone
)

registry.register(
    name="get_exchange_rate",
    description="Get exchange rate between two currencies, optionally for a specific amount.",
    schema={
        "type": "object",
        "properties": {
            "from_currency": {"type": "string"},
            "to_currency":   {"type": "string"},
            "amount":        {"type": "number"}
        },
        "required": ["from_currency", "to_currency"]
    },
    fn=_get_exchange_rate
)

print(f"\n✓ Registry ready — {len(registry.get_tool_schemas())} tools available")

## Step 5 — The Agent

The agent calls `registry.get_tool_schemas()` to discover tools and `registry.execute()` to run them. It never touches tool implementations directly.

In [ ]:
def run_agent(query: str, max_turns: int = 10) -> str:
    """
    Agent that uses the registry for ALL tool access.
    No direct tool calls. Only registry.execute().
    Swap the registry → agent behaviour changes with zero agent code changes.
    """
    messages = [{"role": "user", "content": query}]
    tools    = registry.get_tool_schemas()  # fetched from registry, not hardcoded

    print(f"\n── Query: {query}")

    for _ in range(max_turns):
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1024,
            system="You are a helpful travel and information assistant. Use tools to get accurate information before answering. Be specific.",
            tools=tools,
            messages=messages
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "end_turn":
            answer = next((b.text for b in response.content if hasattr(b, "text")), "")
            print(f"\n── Answer: {answer}")
            return answer

        # All tool calls go via registry.execute() — no direct calls
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = registry.execute(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result
                })
        messages.append({"role": "user", "content": tool_results})

    return "Max turns reached"

print("✓ Agent defined")

## Step 6 — Run Queries

Three queries, each requiring different combinations of tools. The agent decides which tools to call — the registry handles the rest.

In [ ]:
# Single tool
run_agent("What is the weather like in Tokyo right now?")

In [ ]:
# Two tools: weather + timezone (meeting planning)
run_agent(
    "I'm in London planning a call with someone in New York. "
    "What's the weather in both cities and what time difference should I account for?"
)

In [ ]:
# Two tools: currency + unit conversion (travel planning)
run_agent(
    "I have £2,000 for a trip to Tokyo. How much is that in Japanese Yen? "
    "And if the hotel is 3km from the station, how far is that in miles?"
)

In [ ]:
# Show the complete audit trail
registry.show_call_log()

## Key Takeaways

| Point | Why it matters |
|-------|---------------|
| N×M → N+M | Register a tool once. All agents benefit. No per-agent integration work. |
| Agent is decoupled from implementations | Change `_get_weather` to call a real API → zero agent code changes. |
| Call log = free observability | Every tool call recorded with inputs and outputs. Security and compliance audit-ready. |
| Errors as strings, not exceptions | Unknown tools and tool failures return string errors. Agents read them and decide. No crashes. |
| Real MCP goes further | The server runs in a separate process, possibly on a different machine. The concept is identical to what you built here. |

---

**Next lab:** Lab 9 — Orchestrator Fan-Out  
A supervisor that decomposes tasks, delegates to specialist workers, and synthesises results.